# Boletín 7: Capítulos 3.3 y 3.4
## Validación, Generalización y Métricas de Evaluación

**Objetivo:** Practicar la validación correcta de modelos (evitando overfitting y data leakage) y aprender a calcular e interpretar las métricas de evaluación adecuadas según el tipo de problema.

**Instrucciones:**
- 10 ejercicios ordenados por dificultad
- 🟢 = Básico | 🟡 = Intermedio | 🔴 = Avanzado
---

In [ ]:
# Ejecuta esta celda PRIMERO para cargar todas las librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression, load_iris
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV, learning_curve)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_curve, roc_auc_score,
                             mean_absolute_error, mean_squared_error, r2_score)

np.random.seed(42)
print("Librerías cargadas correctamente")

## 🟢 Ejercicio 1: Reconocer overfitting y underfitting a partir de métricas

Un compañero ha entrenado 3 modelos distintos sobre el mismo dataset y te muestra los resultados en train y en test. Tu tarea es **diagnosticar** cada modelo.

A continuación tienes los resultados de los 3 modelos. Para cada uno:

1. **Indica** si el modelo sufre **overfitting**, **underfitting** o tiene **buena generalización**. Justifica tu respuesta en 1-2 frases.
2. **Propón** una acción concreta para mejorar cada modelo problemático (por ejemplo: "usar un modelo más complejo", "añadir regularización", "conseguir más datos", etc.).

| Modelo | Accuracy en Train | Accuracy en Test |
|--------|-------------------|------------------|
| A      | 99.8%             | 62.3%            |
| B      | 58.1%             | 55.7%            |
| C      | 91.2%             | 89.5%            |

Responde con `print()` en la celda de abajo.

In [ ]:
# Escribe aquí tu diagnóstico para cada modelo (A, B y C)


## 🟢 Ejercicio 2: División train/test con stratify

En este ejercicio vas a comprobar por qué es importante usar `stratify` cuando las clases están desbalanceadas.

**Pasos a seguir:**

1. Genera un dataset desbalanceado ejecutando el código de la celda de abajo (ya está escrito).
2. Haz un `train_test_split` con `test_size=0.3` y `random_state=42` **SIN** usar `stratify`. Imprime cuántas muestras de cada clase hay en train y en test usando `np.unique(y_train, return_counts=True)`.
3. Haz otro `train_test_split` con los mismos parámetros **PERO** añadiendo `stratify=y`. Imprime las mismas cuentas.
4. **Responde**: ¿En cuál de los dos splits las proporciones de las clases se mantienen mejor respecto al dataset original? ¿Por qué es importante esto en clasificación?

In [ ]:
# Dataset desbalanceado (NO modificar esta línea)
X, y = make_classification(n_samples=300, n_features=5, n_classes=2,
                           weights=[0.85, 0.15], random_state=42)

# Comprueba la distribución original
print("Distribución original:", np.unique(y, return_counts=True))

# --- Paso 2: Split SIN stratify ---
# Tu código aquí


# --- Paso 3: Split CON stratify ---
# Tu código aquí


# --- Paso 4: Responde ---


## 🟡 Ejercicio 3: Pipeline vs enfoque manual (data leakage con StandardScaler)

En este ejercicio vas a ver un caso **real** de data leakage al preprocesar datos. `StandardScaler` **calcula estadísticos** (media y desviación típica) a partir de los datos — y si los calcula sobre todo el dataset **incluyendo el test**, está "filtrando" información del test al entrenamiento.

**Contexto:** Tienes un dataset de regresión y quieres aplicar `StandardScaler` + `Ridge` (modelo lineal regularizado).

**Pasos a seguir:**

### Parte A — Enfoque INCORRECTO (data leakage real)
1. Con el dataset generado abajo, aplica `StandardScaler()` a **TODO** `X` usando `fit_transform`. Esto calcula la media y std con los 200 datos.
2. **Después** haz el `train_test_split(test_size=0.3, random_state=42, shuffle=False)` sobre los datos ya escalados.
3. Entrena un `Ridge(alpha=20.0)` y calcula el **RMSE** en test. Imprímelo.

### Parte B — Enfoque CORRECTO (con Pipeline)
4. Haz **primero** el `train_test_split(test_size=0.3, random_state=42, shuffle=False)` sobre el `X` original (sin escalar).
5. Crea un `Pipeline` con dos pasos: `('scaler', StandardScaler())` y `('ridge', Ridge(alpha=20.0))`.
6. Entrena el Pipeline con `X_train, y_train` y calcula el **RMSE** en test. Imprímelo.

### Parte C — Reflexión
7. **Responde**: ¿Por qué el enfoque A produce data leakage real? ¿Qué información del test se ha "filtrado" al modelo?

In [ ]:
from sklearn.preprocessing import StandardScaler

# Dataset de regresión (NO modificar la base)
X, y = make_regression(n_samples=200, n_features=3, noise=15, random_state=42)

# Cambio suave en la distribución del tramo final (test) para que se note el efecto
X[140:, 0] = X[140:, 0] * 2.0 + 1.0

# --- Parte A: Enfoque INCORRECTO (data leakage) ---
# Aplica StandardScaler a TODO X antes del split
# Tu código aquí


# --- Parte B: Enfoque CORRECTO (Pipeline) ---
# Haz el split primero, luego usa Pipeline con StandardScaler
# Tu código aquí



# --- Parte C: Responde ---

## 🟡 Ejercicio 4: Validación cruzada (Cross-Validation)

En este ejercicio vas a comparar la evaluación de un modelo con un **único split** frente a **validación cruzada con 5 folds**.

**Pasos a seguir:**

1. Con el dataset generado abajo, haz un `train_test_split(test_size=0.2, random_state=42, stratify=y)`. Entrena un `LogisticRegression(max_iter=1000, random_state=42)` y calcula la accuracy en test. Imprímela.
2. Ahora, usa `cross_val_score` con el **mismo modelo** sobre el dataset completo (`X, y`) y `cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`. Imprime los 5 scores individuales, la **media** y la **desviación estándar**.
3. **Responde**: ¿Cuál de las dos evaluaciones es más fiable y por qué? ¿Qué te dice la desviación estándar de los folds?

In [ ]:
# Dataset de clasificación (NO modificar)
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_redundant=2, random_state=42)

# --- Paso 1: Evaluación con un único split ---
# Tu código aquí


# --- Paso 2: Evaluación con cross-validation (5-Fold) ---
# Tu código aquí


# --- Paso 3: Responde ---


## 🟡 Ejercicio 5: Búsqueda de hiperparámetros con GridSearchCV

En este ejercicio vas a usar `GridSearchCV` dentro de un `Pipeline` para buscar el mejor valor del hiperparámetro `C` de una `LogisticRegression`.

**Pasos a seguir:**

1. Con el dataset generado abajo, haz un `train_test_split(test_size=0.2, random_state=42, stratify=y)`.
2. Crea un `Pipeline` con dos pasos: `('scaler', StandardScaler())` y `('clf', LogisticRegression(max_iter=1000, random_state=42))`.
3. Define un diccionario de hiperparámetros a probar: `{'clf__C': [0.01, 0.1, 1, 10, 100]}`. Recuerda: dentro de un Pipeline, el nombre del parámetro se forma como `'nombre_paso__nombre_parametro'`.
4. Crea un `GridSearchCV` con el Pipeline, el diccionario de hiperparámetros, `cv=5` y `scoring='accuracy'`. Entrénalo con `X_train, y_train`.
5. Imprime: el **mejor valor de C** (`grid.best_params_`), el **mejor score en validación** (`grid.best_score_`), y la **accuracy en test** usando `grid.score(X_test, y_test)`.
6. **Responde**: ¿Por qué es importante que el `StandardScaler` esté dentro del Pipeline y no fuera?

In [ ]:
# Dataset de clasificación (NO modificar)
X, y = make_classification(n_samples=600, n_features=10, n_informative=6,
                           n_redundant=2, random_state=42)

# --- Paso 1: Split ---
# Tu código aquí


# --- Pasos 2-4: Pipeline + GridSearchCV ---
# Tu código aquí


# --- Paso 5: Resultados ---
# Tu código aquí


# --- Paso 6: Responde ---


## 🟡 Ejercicio 6: Matriz de confusión — construir e interpretar

En este ejercicio vas a construir una matriz de confusión a partir de predicciones ya hechas y a interpretar cada celda.

**Contexto:** Un modelo de detección de enfermedades ha evaluado a 20 pacientes. Abajo tienes las etiquetas reales (`y_real`) y las predicciones del modelo (`y_pred`), donde `1 = enfermo` y `0 = sano`.

**Pasos a seguir:**

1. Calcula la matriz de confusión con `confusion_matrix(y_real, y_pred)`. Imprímela.
2. Extrae los 4 valores individuales (TN, FP, FN, TP) usando `cm.ravel()` e imprime cada uno con su significado. Por ejemplo: `"TP = 5 → 5 enfermos detectados correctamente"`.
3. Visualiza la matriz de confusión con `ConfusionMatrixDisplay`. Usa `display_labels=['Sano', 'Enfermo']`.
4. **Responde**: En este contexto médico, ¿cuál es el error más peligroso, un FP (diagnosticar como enfermo a alguien sano) o un FN (diagnosticar como sano a alguien enfermo)? ¿Por qué?

In [ ]:
# Datos del modelo de detección de enfermedades (NO modificar)
# 1 = enfermo, 0 = sano
y_real = np.array([0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1])
y_pred = np.array([0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1])

# --- Paso 1: Matriz de confusión ---
# Tu código aquí


# --- Paso 2: Extraer TN, FP, FN, TP ---
# Tu código aquí


# --- Paso 3: Visualización ---
# Tu código aquí


# --- Paso 4: Responde ---


## 🟡 Ejercicio 7: Accuracy, Precision, Recall y F1 en datos desbalanceados

En este ejercicio vas a comprobar que la **accuracy es engañosa** cuando las clases están desbalanceadas.

**Contexto:** Un modelo de detección de fraude evalúa 200 transacciones. De esas 200, solo 10 son fraudulentas (clase 1) y 190 son legítimas (clase 0). El modelo predice **siempre** que la transacción es legítima (clase 0).

**Pasos a seguir:**

1. Calcula la **accuracy** con `accuracy_score`. Imprímela. ¿Parece un buen resultado?
2. Calcula la **precision** con `precision_score(y_real, y_pred, zero_division=0)`. Imprímela.
3. Calcula el **recall** con `recall_score`. Imprímela.
4. Calcula el **F1-score** con `f1_score(y_real, y_pred, zero_division=0)`. Imprímelo.
5. Ejecuta `classification_report(y_real, y_pred, target_names=['Legítima', 'Fraude'], zero_division=0)` e imprime el resultado.
6. **Responde**: ¿Por qué la accuracy del 95% es **engañosa**? ¿Qué métrica refleja mejor que el modelo es **inútil** para detectar fraude? Justifica tu respuesta.

In [ ]:
# Datos del modelo de detección de fraude (NO modificar)
y_real = np.array([0]*190 + [1]*10)       # 190 legítimas + 10 fraudes
y_pred = np.array([0]*200)                # El modelo siempre predice "legítima"

# --- Paso 1: Accuracy ---
# Tu código aquí


# --- Paso 2: Precision ---
# Tu código aquí


# --- Paso 3: Recall ---
# Tu código aquí


# --- Paso 4: F1-Score ---
# Tu código aquí


# --- Paso 5: Classification Report ---
# Tu código aquí


# --- Paso 6: Responde ---


## 🔴 Ejercicio 8: Curva ROC y AUC

En este ejercicio vas a construir una curva ROC y calcular el AUC a partir de las probabilidades de un modelo.

**Contexto:** Un modelo de clasificación binaria ha evaluado 10 muestras. Tienes las etiquetas reales y las **probabilidades** que el modelo asigna a la clase positiva (clase 1).

**Pasos a seguir:**

1. Usa `roc_curve(y_real, y_proba)` para obtener `fpr`, `tpr` y `thresholds`. Imprime los tres arrays.
2. Calcula el AUC con `roc_auc_score(y_real, y_proba)`. Imprímelo.
3. Dibuja la curva ROC: eje X = FPR, eje Y = TPR. Añade:
   - Una línea diagonal punteada (representa un modelo aleatorio, AUC = 0.5).
   - El valor del AUC en el título o la leyenda del gráfico.
   - Etiquetas en los ejes: "False Positive Rate (FPR)" y "True Positive Rate (TPR)".
4. **Responde**: ¿Es este modelo bueno o malo? Justifica tu respuesta usando el valor del AUC y la tabla de referencia: AUC > 0.9 excelente, 0.8-0.9 bueno, 0.7-0.8 aceptable, 0.6-0.7 pobre, 0.5-0.6 aleatorio.

In [ ]:
# Datos del modelo (NO modificar)
y_real =  np.array([0, 1, 0, 1, 1, 0, 1, 0, 0, 1])
y_proba = np.array([0.15, 0.85, 0.25, 0.78, 0.92, 0.30, 0.88, 0.10, 0.45, 0.70])

# --- Paso 1: Calcular fpr, tpr, thresholds ---
# Tu código aquí


# --- Paso 2: Calcular AUC ---
# Tu código aquí


# --- Paso 3: Dibujar la curva ROC ---
# Tu código aquí


# --- Paso 4: Responde ---


## 🔴 Ejercicio 9: Métricas de regresión — MAE, RMSE y R²

En este ejercicio vas a calcular e interpretar las tres métricas principales de regresión.

**Contexto:** Un modelo predice el precio de viviendas (en miles de euros). Abajo tienes los precios reales y los predichos para 8 viviendas.

**Pasos a seguir:**

1. Calcula el **MAE** (Mean Absolute Error) usando `mean_absolute_error`. Imprímelo con la frase: `"MAE = X.XX miles de euros -> En promedio, el modelo se equivoca en X.XX miles de euros"`.
2. Calcula el **RMSE** (Root Mean Squared Error). Para ello, calcula primero el MSE con `mean_squared_error` y después aplica `np.sqrt()`. Imprímelo con la frase: `"RMSE = X.XX miles de euros"`.
3. Calcula el **R²** con `r2_score`. Imprímelo con la frase: `"R² = X.XX -> El modelo explica el XX% de la variabilidad de los precios"`.
4. **Responde**: ¿Por qué el RMSE es mayor que el MAE? ¿Qué nos dice eso sobre los errores del modelo? Pista: fíjate en la vivienda 7 (precio real: 480, predicho: 420).

In [ ]:
# Precios de viviendas en miles de € (NO modificar)
y_real = np.array([200, 350, 150, 420, 300, 180, 480, 275])
y_pred = np.array([210, 330, 160, 400, 310, 175, 420, 260])

# --- Paso 1: MAE ---
# Tu código aquí


# --- Paso 2: RMSE ---
# Tu código aquí


# --- Paso 3: R² ---
# Tu código aquí


# --- Paso 4: Responde ---


## 🔴 Ejercicio 10: Elegir la métrica correcta según el caso de negocio

En este ejercicio final vas a aplicar todo lo aprendido para elegir la métrica adecuada en diferentes escenarios reales.

**Para cada caso de negocio**, responde a estas 3 preguntas:
- **a)** ¿Qué tipo de error es más grave: un falso positivo (FP) o un falso negativo (FN)?
- **b)** ¿Qué métrica usarías como métrica principal? Elige entre: Accuracy, Precision, Recall, F1, ROC-AUC, MAE, RMSE, R².
- **c)** Justifica tu elección en 1-2 frases.

---

**Caso 1 — Diagnóstico de cáncer:** Un modelo predice si un paciente tiene cáncer (positivo) o no (negativo). El hospital quiere detectar **todos** los casos posibles, aunque eso genere algunas falsas alarmas.

**Caso 2 — Filtro de spam:** Un modelo clasifica correos como spam (positivo) o legítimo (negativo). Que un spam pase no es grave, pero **perder un correo legítimo** marcándolo como spam sí es un problema serio.

**Caso 3 — Predicción de temperatura:** Un modelo meteorológico predice la temperatura máxima diaria en grados Celsius. Quieres saber cuánto se equivoca de media.

**Caso 4 — Detección de fraude bancario:** Un modelo clasifica transacciones como fraudulentas (positivo) o legítimas (negativo). El dataset tiene 99.5% de transacciones legítimas y 0.5% de fraudes.

In [ ]:
# Responde aquí a los 4 casos con print()


---

## Resumen de Conceptos Clave (Capítulos 3.3 y 3.4)

### Cap. 3.3 — Validación y Generalización
| Concepto | Descripción |
|----------|-------------|
| **Overfitting** | Train alto, test bajo. El modelo memoriza en vez de aprender |
| **Underfitting** | Train bajo, test bajo. El modelo es demasiado simple |
| **Stratify** | Mantiene las proporciones de clases al dividir datos |
| **Pipeline** | Encapsula preprocesamiento + modelo para evitar data leakage |
| **Cross-Validation** | Evalúa el modelo en múltiples splits (K-Fold). Reporta media +/- std |
| **GridSearchCV** | Busca el mejor hiperparámetro combinando grid + cross-validation |

### Cap. 3.4 — Métricas de Evaluación
| Métrica | Fórmula / Uso |
|---------|---------------|
| **Accuracy** | Aciertos / Total. Solo fiable con clases balanceadas |
| **Precision** | TP / (TP + FP). Prioriza cuando los FP son costosos |
| **Recall** | TP / (TP + FN). Prioriza cuando los FN son inaceptables |
| **F1-Score** | Media armónica de Precision y Recall |
| **ROC-AUC** | Evalúa el modelo en todos los thresholds. AUC > 0.9 = excelente |
| **MAE** | Error absoluto medio. Robusto a outliers |
| **RMSE** | Raíz del error cuadrático medio. Penaliza errores grandes |
| **R²** | Porcentaje de variabilidad explicada. 1.0 = perfecto, 0 = como la media |